# Configure Grafana Alerting

**Module 8: Alerting with Grafana** · [Quiz](quiz08_alerting_rules.ipynb) · [Course README](../../README.md)

In this Lab you build an alert that emails you when the CPU of a host stays busy. You look at how the DOG stack stores host CPU in Doris and write the query that the alert evaluates. Then you set up the three parts of Grafana alerting: a contact point, which says how to send a notification; a notification policy, which says which alerts go to that contact point; and the alert rule, which says what Grafana evaluates and when the alert fires. Finally you make the CPU busy and watch the alert fire and resolve, and you stop the host metrics to see what Grafana does when data stops arriving.

Grafana sends its emails to Mailpit, a local mail server that the DOG stack starts with its other containers. No email leaves your computer.

Plan about 40 minutes. Two steps wait for Grafana: about 4 minutes while the CPU is busy, and about 3 minutes while the host metrics are stopped. Sections 4 to 6 take place in the Grafana web page; run the cells in order around them. You need the DOG stack from Module 1; the demo can run or not.

### Initialize the Lab

Run this cell once before using `lab.sql(...)`, `lab.shell(...)`, or the alerting helpers.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)

## 1. Check the host metrics

Every 15 seconds, the DOG Collector measures the CPU of the machine that runs Docker and writes the result to `otel_metrics_gauge` as the metric `system.cpu.utilization`. With Docker Desktop on macOS or Windows, that machine is Docker Desktop's Linux virtual machine; on Linux, it is your computer. The Collector labels the rows with the name of the host (`host.name` in `resource_attributes`), so that an alert can say which host is busy. The next cell waits until the rows of the last minute are there.

In [ ]:
host_metrics = lab.wait_for_rows(
    """
    SELECT CAST(resource_attributes['host.name'] AS STRING) AS host,
           COUNT(DISTINCT CAST(attributes['cpu'] AS STRING)) AS cpu_cores,
           COUNT(DISTINCT timestamp) AS samples,
           MAX(timestamp) AS latest_sample
    FROM otel_metrics_gauge
    WHERE metric_name = 'system.cpu.utilization'
      AND scope_name LIKE '%hostmetricsreceiver%'
      AND resource_attributes['host.name'] IS NOT NULL
      AND timestamp >= UTC_TIMESTAMP() - INTERVAL 1 MINUTE
    GROUP BY host
    """,
    title="Host CPU samples from the last minute",
    hint="The DOG Collector writes these rows every 15 seconds. Run lab.prepare_environment() in Module 1 "
         "again: it starts the DOG stack, or applies the course's Collector settings to a running stack. "
         "Then rerun this cell.",
)

**Expected result**

One row, usually at once: your Docker host, such as `docker-desktop` with Docker Desktop, or your computer's name on Linux. `cpu_cores` is the number of CPUs of that host (Docker Desktop's settings show it), `samples` is 3 or 4, one every 15 seconds, and `latest_sample` is within half a minute of the current time in UTC.

## 2. Choose the rows that measure host CPU

### Every source of `system.cpu.utilization`

Other programs report a metric with the same name. The next cell lists every source that reported `system.cpu.utilization` in the last two minutes; some of them report only once a minute.

In [ ]:
lab.sql("""
SELECT scope_name, scope_version, service_name,
       CAST(resource_attributes['host.name'] AS STRING) AS host,
       COUNT(DISTINCT CAST(attributes['cpu'] AS STRING)) AS cpu_cores,
       COUNT(DISTINCT CAST(attributes['state'] AS STRING)) AS states,
       COUNT(*) AS row_count
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 2 MINUTE
GROUP BY scope_name, scope_version, service_name, host
ORDER BY scope_name, service_name
""", title="Sources of system.cpu.utilization in the last two minutes");

**Expected result**

Four rows while the demo runs, and only the DOG Collector's row without it. The demo's Collector and the Python services report once a minute, the DOG Collector every 15 seconds, so `row_count` differs:

| `scope_name` | `scope_version` | `service_name` | `host` | Source |
| --- | --- | --- | --- | --- |
| `github.com/.../hostmetricsreceiver/internal/scraper/cpuscraper` | `0.144.0` | (empty) | your host | The DOG Collector's host metrics |
| the same | `0.120.1` | (empty) | NULL | The demo Collector's host metrics |
| `opentelemetry.instrumentation.system_metrics` | `0.51b0` | `load-generator`, `recommendation` | NULL | The system metrics of two Python services of the demo, one row each |

All four count the same cores: the containers share the kernel of the Docker host. A rule that filters only on the metric name mixes several copies of the same CPUs, and only the first source says which host its rows describe. The alert reads one source per host: the rows of a Collector's host metrics receiver (`scope_name LIKE '%hostmetricsreceiver%'`) that carry a host name. The DOG Collector adds the name with its `resourcedetection` processor, set up in `environments/dog-stack/collector-host-metrics.yml`.

### One sample of one core

Each row is the share of time that one core spent in one state during the last 15 seconds. The next cell shows the newest sample of the core `cpu0`.

In [ ]:
lab.sql("""
SELECT timestamp, CAST(attributes['state'] AS STRING) AS state, ROUND(value, 4) AS share_of_time
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND scope_name LIKE '%hostmetricsreceiver%'
  AND resource_attributes['host.name'] IS NOT NULL
  AND attributes['cpu'] = 'cpu0'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 1 MINUTE
ORDER BY timestamp DESC, share_of_time DESC
LIMIT 8
""", title="The newest sample of cpu0");

**Expected result**

Eight rows with the same `timestamp`, one per state: `idle`, `user`, `system`, `softirq`, `wait`, `nice`, `interrupt`, and `steal`. Their shares add up to 1. `idle` is the largest share, often around 0.9; the core was busy for the rest of the time.

### Why a filter on the metric name alone fires on idle hosts

A rule that filters only on `metric_name` and compares every row with a threshold, such as 0.98, takes each of these rows for a measure of how busy the CPU is. The next cell counts, per state, the rows of the last minute and how many of them are above 0.9.

In [ ]:
lab.sql("""
SELECT CAST(attributes['state'] AS STRING) AS state,
       COUNT(*) AS row_count,
       ROUND(AVG(value), 3) AS avg_share,
       SUM(IF(value > 0.9, 1, 0)) AS rows_above_0_9
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 1 MINUTE
GROUP BY CAST(attributes['state'] AS STRING)
ORDER BY avg_share DESC
""", title="Rows of system.cpu.utilization by state in the last minute");

**Expected result**

`idle` comes first, with an average share around 0.9, and nearly every row above 0.9 is an `idle` row; the other states have none. (`irq` is the Python services' name for interrupts.) Such a rule fires because the CPUs are idle: the quieter the host, the more rows pass. And since each row is a core and a state of some source, its alerts cannot say which host is busy.

## 3. The alert query

The query that the alert evaluates turns these rows into one number per host:

- `attributes['state'] = 'idle'` and `1 - AVG(value)`: the average idle share of the host's cores, over every sample in the window, subtracted from 1. That is the busy share; times 100, it is a percentage.
- `scope_name LIKE '%hostmetricsreceiver%'` and `resource_attributes['host.name'] IS NOT NULL`: one source per host, as section 2 showed.
- `GROUP BY host`: one row per host. Grafana makes an alert instance of each row and labels it with the row's text columns, so a notification names its host (`host=...`).
- `timestamp >= UTC_TIMESTAMP() - INTERVAL 1 MINUTE`: the window ends now, not at the newest row, as it did in earlier Labs. When a host stops reporting, its rows leave the window and the query returns nothing for it, instead of evaluating its last samples again and again. Section 8 shows what Grafana does then. The tables store UTC, so the window starts from `UTC_TIMESTAMP()`.

Run the query now. In section 6, you paste the same SQL into Grafana.

In [ ]:
HOST_CPU_QUERY = """
SELECT CAST(resource_attributes['host.name'] AS STRING) AS host,
       ROUND((1 - AVG(value)) * 100, 1) AS cpu_busy_percent
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND attributes['state'] = 'idle'
  AND scope_name LIKE '%hostmetricsreceiver%'
  AND resource_attributes['host.name'] IS NOT NULL
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 1 MINUTE
GROUP BY host
"""
lab.sql(HOST_CPU_QUERY, title="Busy share of each host over the last minute (the alert query)");

**Expected result**

One row: your host and `cpu_busy_percent`, usually between 3 and 20. Rerun the cell a few times: the value moves with the work of the demo and of Doris, but it stays far below 90, the threshold of the rule in section 6.

## 4. The mail server and a contact point

### Mailpit

Grafana sends email over SMTP. In this course's DOG stack, it sends to the `doris-obs-mailpit` container: Mailpit accepts every message, whatever the address, and shows it in a web inbox. Open the inbox at [http://127.0.0.1:28025](http://127.0.0.1:28025) (or the port in `DORIS_OBS_MAILPIT_PORT`). The next cell shows the SMTP settings that `environments/dog-stack/compose.dog.yml` gives Grafana.

In [ ]:
lab.shell(
    "docker inspect --format '{{range .Config.Env}}{{println .}}{{end}}' doris-obs-grafana | grep '^GF_SMTP_'",
    title="Grafana's SMTP settings",
);

**Expected result**

The command output lists four settings: `GF_SMTP_ENABLED=true`, `GF_SMTP_HOST=doris-obs-mailpit:1025` (the Mailpit container and its SMTP port, on the `doris-obs` network), `GF_SMTP_FROM_ADDRESS`, and `GF_SMTP_FROM_NAME`. Without them, Grafana cannot send email, and the Test of an email contact point fails with an SMTP error. For a real mail server, its address, user, and password go in the same settings, or in the `[smtp]` section of `grafana.ini`.

### Create a contact point and test it

A contact point says how to send a notification: here, by email to one address. In Grafana:

1. Sign in at [http://127.0.0.1:23000/login](http://127.0.0.1:23000/login) with the user `admin` and the password `admin`. Grafana then asks for a new password: choose **Skip**, or set one. This Grafana listens only on your computer, and the cells of this Lab read Grafana without signing in.
2. Open **Alerting** > **Contact points** and choose **+ Create contact point**.
3. **Name**: `doris-course-email`. **Integration**: `Email`, the default. **Addresses**: `oncall@example.com`.
4. Choose **Test**, then **Send test notification**. Grafana reports "Test alert sent." Close the dialog.
5. Choose **Save contact point**.

Then run the next cell. It reads the inbox through Mailpit's API.

In [ ]:
test_email = lab.wait_for_email(
    "TestAlert",
    hint="In the contact point form, choose Test and then Send test notification. If Grafana reports an "
         "SMTP error, check the settings that the previous cell showed, then rerun this cell.",
)

**Expected result**

A table with one email, from `Grafana (DOG course) <grafana@example.com>` to `oncall@example.com`, with the subject `[FIRING:1] TestAlert Grafana`. The inbox in the browser shows the same email. Its text has the summary "Notification test" and the labels `alertname=TestAlert` and `instance=Grafana`.

This email shows that Grafana can deliver to this contact point. It does not show that a rule fires when it should, that its alerts take this route, or that a resolved notification follows. The next sections check each of these.

## 5. Route the alert with a notification policy

A notification policy decides, by their labels, which alerts go to which contact point, and how Grafana groups and times their notifications. An alert that matches no other policy follows the default policy, which sends it to the contact point `grafana-default-email`. Add a policy for the alerts of this course:

1. Open **Alerting** > **Notification policies**. Next to **Default policy**, choose **+ New child policy**.
2. **Matching labels**: label `team`, operator `=`, value `doris-course`.
3. **Contact point**: `doris-course-email`.
4. Turn on **Override grouping**. **Group by** already lists `grafana_folder` and `alertname`: add `host`. Each host then gets a notification of its own.
5. Turn on **Override general timings** and enter **Group wait** `10s`, **Group interval** `1m`, and **Repeat interval** `4h`. Group wait is how long Grafana waits before the first notification of a new group, so that alerts that fire together arrive in one email. Group interval is how long it waits before it notifies about a change in the group, such as an alert that resolves. The defaults, 30 seconds and 5 minutes, suit production; the shorter values save time in this Lab.
6. Choose **Save policy**. The new policy shows `team = doris-course`, "Delivered to doris-course-email", and the grouping and timings that you entered.

The rule of the next section carries the label `team=doris-course`, so its alerts take this route. The contact point decides how to notify; the policy decides which alerts go there.

## 6. Create the alert rule

In Grafana, open **Alerting** > **Alert rules** and choose **+ New alert rule**. The form has five sections.

**1. Enter alert rule name**: `Host CPU busy`. The cells below find the rule by this name.

**2. Define query and alert condition**

- Query **A** uses the `Doris` data source. Switch its editor from **Builder** to **Code**, keep **Format** at `Table`, and paste the SQL of `HOST_CPU_QUERY` from section 3. Choose **Run query**: the table below the editor shows `{host="..."}` and the busy percentage.
- Under **Expressions**, Grafana has added a **Reduce** expression, B, and a **Threshold** expression, C. A query in Table format already returns one number per host, and Grafana warns that the Reduce operation is not needed: delete expression B with its trash icon.
- In expression **C (Threshold)**, set **Input** to `A`, keep **IS ABOVE**, and enter `90`. C is the alert condition. Choose **Preview**: your host shows `0` and `Normal`.

**3. Set evaluation behavior**

- **Folder**: choose **+ New folder** and create `Doris course`.
- **Evaluation group and interval**: choose **+ New evaluation group**, name it `host-cpu`, and set the interval to `30s`. Grafana evaluates every rule of the group every 30 seconds.
- **Pending period**: `1m`. The condition must hold at every evaluation for a minute before the alert fires. One evaluation below the threshold returns the instance to Normal, and the minute starts again. A production rule often waits longer, such as 10 minutes, so that short peaks do not wake anyone; this Lab uses a minute so that you can watch the alert fire.
- Expand **Configure no data and error handling** and keep the defaults: **Alert state if no data or all values are null** is `No Data`, and **Alert state if execution error or timeout** is `Error`. Section 8 shows the first one at work.

**4. Configure labels and notifications**

- **Labels**: choose **Add labels**, add `team` = `doris-course` and `severity` = `warning` (type a new key or value and press Enter), and choose **Save**.
- **Notifications**: choose **Use notification policy**, then **Preview routing**. The alert instance goes to the policy `team = doris-course`, "Delivered to doris-course-email".

**5. Configure notification message**

- **Summary**: `CPU of {{ $labels.host }} is {{ $values.A }}% busy`
- **Description**: `The CPU cores of {{ $labels.host }} have been more than 90% busy on average for at least 1 minute.`

`{{ $labels.host }}` is the host label of the instance, and `{{ $values.A }}` is the value of query A at the evaluation. Choose **Save rule and exit**. The rule list shows `Doris course > host-cpu` with `1 normal`. The next cell reads the rule through Grafana's API.

In [ ]:
lab.alert_instances("Host CPU busy");

**Expected result**

One instance, `host=...` with your host, in the state `Normal`, with a summary such as "CPU of docker-desktop is 8.5% busy". The title gives the time of the last evaluation, at most 30 seconds ago, and the health `ok`. Right after you save the rule, the cell can report "No instances yet" instead: Grafana evaluates a new rule at the next 30-second tick of its group, so rerun the cell. If the cell stops with "Grafana has no alert rule named 'Host CPU busy'", check the name of the rule.

## 7. Make the CPU busy

The next cell starts a container that keeps every CPU of the Docker host busy: one `yes` loop per CPU, at the lowest priority (`nice -n 19`), so that Doris, Grafana, and the Collectors still get the CPU time that they ask for. Your computer may run its fans. The container stops by itself after 5 minutes, and a later cell stops it as soon as the alert has fired.

In [ ]:
from datetime import datetime, timezone

load_started = datetime.now(timezone.utc)
lab.shell(r"""
set -euo pipefail

docker rm -f doris-obs-cpu-load >/dev/null 2>&1 || true
docker run -d --rm --name doris-obs-cpu-load busybox:1.37.0 \
  sh -c 'for cpu in $(seq "$(nproc)"); do nice -n 19 yes > /dev/null & done; sleep 300'

docker ps --filter name=doris-obs-cpu-load --format '{{.Names}} {{.Status}}'
""", title="Keep every CPU of the Docker host busy");

**Expected result**

The command log ends with `doris-obs-cpu-load Up ...`. The first run downloads the `busybox` image, about 2 MB.

### Watch the alert fire

The next cell reads the rule every 5 seconds until its instance is `Alerting`, and lists every change on the way.

In [ ]:
lab.wait_for_alert_state(
    "Host CPU busy", "Alerting",
    hint="Check that the load runs with docker ps --filter name=doris-obs-cpu-load: it stops after 5 minutes, "
         "and the previous cell starts it again. Check also the threshold 90 and the pending period 1m of "
         "the rule.",
    timeout_seconds=300,
);

**Expected result**

After two to three minutes, the table of changes shows the instance `Normal`, then `Pending`, then `Alerting`, each at the evaluation that changed it:

- About a minute after the load starts, most of the query's one-minute window is busy, and the busy share passes 90%. Run `lab.sql(HOST_CPU_QUERY)` in a new cell to watch it climb.
- The first evaluation above 90 makes the instance `Pending`.
- Two evaluations, one minute, later, it is `Alerting`. Its summary gives the busy share at that evaluation, about 95%.

In [ ]:
firing_email = lab.wait_for_email(
    "[FIRING:1] Host CPU busy",
    after=load_started,
    hint="Check the notification policy of section 5 (matching label team = doris-course, contact point "
         "doris-course-email) and the label team=doris-course of the rule.",
)

**Expected result**

An email about 10 seconds after the alert fired, the group wait of the policy, with a subject such as `[FIRING:1] Host CPU busy Doris course docker-desktop (warning doris-course)`: the number of firing instances, the values of the policy's group-by labels, and the rule's other labels. Its text gives the summary "CPU of ... is ...% busy", the values (`A=` the busy share, and `C=1`, the condition is true), all the labels with `host`, and links to the rule and to a form that silences it.

### Stop the load and watch the alert resolve

In [ ]:
lab.shell("docker rm -f doris-obs-cpu-load", title="Stop the CPU load");
lab.wait_for_alert_state(
    "Host CPU busy", "Normal",
    hint="Check that docker ps --filter name=doris-obs-cpu-load lists no container, then rerun this cell.",
    timeout_seconds=180,
);

**Expected result**

The load stops at once. Within about half a minute, idle samples pull the busy share of the window below 90%, and the next evaluation returns the instance to `Normal`, usually within a minute of the cell's start.

In [ ]:
resolved_email = lab.wait_for_email(
    "[RESOLVED] Host CPU busy",
    after=load_started,
    hint="Grafana notifies about the resolved alert at the next group interval of the policy, 1 minute. "
         "Rerun this cell.",
)

**Expected result**

An email with a subject such as `[RESOLVED] Host CPU busy Doris course docker-desktop (warning doris-course)`. Grafana sends it at the next group interval of the policy: one minute after the firing email when the alert resolved within that minute, as it usually does here.

With the firing email, it shows the rule, the route, and the contact point working once, for one host and one load. It does not show how the rule behaves with many hosts, or with a load that comes and goes around the threshold.

## 8. When the host stops reporting

The query reads only the last minute. When the host stops reporting, the query returns no rows, and there is no value left to compare with 90. The rule's setting **Alert state if no data or all values are null** decides what Grafana does then. With the default, `No Data`, the rule reports no data, and Grafana sends a separate alert, `DatasourceNoData`, through the same notification policy. The next cells stop the DOG Collector for about two minutes to show it. While it is stopped, no new telemetry reaches Doris.

In [ ]:
collector_stopped = datetime.now(timezone.utc)
lab.shell("docker stop doris-obs-otel-collector", title="Stop the DOG Collector");
lab.wait_for_alert_state(
    "Host CPU busy", "NoData",
    hint="Check that docker ps --filter name=doris-obs-otel-collector lists no container, and that the rule "
         "keeps the default no data handling of section 6.",
    timeout_seconds=240,
);

**Expected result**

About a minute after the Collector stops, once its last samples have left the window, the rule's health becomes `nodata`, and a new instance, `datasource_uid=doris, ref_id=A`, is `NoData`: it names the data source and the query, not a host. The instance `host=...` is still listed, as `Normal`, after that evaluation, and disappears at the next one: Grafana resolves an instance whose series has been missing from the query result for two evaluations.

In [ ]:
no_data_email = lab.wait_for_email(
    "DatasourceNoData",
    after=collector_stopped,
    hint="Check that the rule keeps the default no data handling and the label team=doris-course.",
)

**Expected result**

An email about 10 seconds after the rule reported no data, with the subject `[FIRING:1] DatasourceNoData Doris course (doris A Host CPU busy warning doris-course)`. Its labels name the rule (`rulename`), the data source (`datasource_uid=doris`), and the query (`ref_id=A`), but no host: without rows, there is no host to name, and the summary shows `[no value]`.

The rule reports no data only when its query returns nothing at all. If one of several hosts stopped reporting, the query would still return the others: the silent host's instance would disappear after two evaluations, without a notification. The exercise at the end of the Lab writes a query for that case.

In [ ]:
lab.shell("docker start doris-obs-otel-collector", title="Start the DOG Collector again");
lab.wait_for_alert_state(
    "Host CPU busy", "Normal",
    hint="Check that the DOG Collector runs with docker ps --filter name=doris-obs-otel-collector, rerun the "
         "cell of section 1, then rerun this cell.",
    timeout_seconds=180,
);

**Expected result**

The host's samples reach Doris again within half a minute. The next evaluations bring back the instance `host=...` as `Normal` and remove the `NoData` instance; the cell ends when only `Normal` instances are left, usually within two minutes. A `[RESOLVED] DatasourceNoData` email follows a few minutes later.

## 9. The rule as code

Grafana can export a rule in the format of its provisioning files. The next cell shows your rule in that format.

In [ ]:
rule_yaml = lab.export_alert_rule("Host CPU busy")

**Expected result**

YAML that starts with `apiVersion: 1` and the group `host-cpu` in the folder `Doris course`, with `interval: 30s`. The rule has your query under `data` (`refId: A`, `rawSql`), the threshold expression C (`type: gt` with `90`), then `noDataState: NoData`, `execErrState: Error`, `for: 1m`, the two annotations, and the two labels: every setting of the form.

Keep such a file in version control to rebuild the rule in another Grafana. Grafana loads the files in its `provisioning/alerting` directory at startup and creates their rules; the web page shows those rules as provisioned and does not let you edit them, so the file stays the one source. Contact points and notification policies have export actions on their pages too.

## 10. Independent exercise

Section 8 showed that a host that stops reporting goes unnoticed while other hosts report. Write a query that returns, for each host whose host metrics arrived in the last hour, the number of seconds since its newest sample, so that a rule could alert when it is above 120. Use the source filters of the alert query. Expect one row: your host, with a few seconds up to about half a minute.

Write and run your code in the next cell, then expand the reference solution.

<details>
<summary>Reference solution (expand after completing the exercise)</summary>

```python
lab.sql("""
SELECT CAST(resource_attributes['host.name'] AS STRING) AS host,
       MAX(timestamp) AS latest_sample,
       TIMESTAMPDIFF(SECOND, MAX(timestamp), UTC_TIMESTAMP()) AS seconds_since_latest_sample
FROM otel_metrics_gauge
WHERE metric_name = 'system.cpu.utilization'
  AND scope_name LIKE '%hostmetricsreceiver%'
  AND resource_attributes['host.name'] IS NOT NULL
  AND timestamp >= UTC_TIMESTAMP() - INTERVAL 1 HOUR
GROUP BY host
""", title="Seconds since the newest host-metrics sample of each host");
```

The window is an hour, not a minute: a host that stopped reporting five minutes ago still has rows in it, so it still has a row in the result, with `seconds_since_latest_sample` near 300. A rule on this query with the condition IS ABOVE 120 fires for that host, by name. After an hour without samples, the host leaves the result too; to cover longer silences, use a longer window or compare with a list of the hosts that should report.

</details>

## Completion criteria

- You can explain why a row of `system.cpu.utilization` does not measure how busy a host is, and what the alert query computes instead.
- The Mailpit inbox has the test email, and the firing and resolved emails of `Host CPU busy`, delivered through the policy `team = doris-course`.
- You can say what the pending period, the group wait, and the no data setting did to what you saw.
- The exported YAML of the rule shows the query, the threshold 90, and the pending period `1m`.

## Clean up

The CPU load container is gone; it stops after 5 minutes at the latest. The rule, the policy, and the contact point stay in Grafana, where the rule keeps watching the host. `lab.teardown()` deletes them with Grafana's volume, and `lab.teardown(keep_data=True)` keeps them. To delete them now, delete the rule in **Alert rules**, the child policy in **Notification policies**, and then the contact point in **Contact points**: Grafana does not delete a contact point that a policy still uses.

## Next

Open [Quiz 8: Alerting with Grafana](quiz08_alerting_rules.ipynb). This is the last module of the course: the [course README](../../README.md) explains how to stop and remove the whole environment.